In [2]:
import fitz  # PyMuPDF
from tqdm.auto import tqdm
from pathlib import Path # Using pathlib for robust path handling


In [3]:
document_titles = [
    "Condition of use",
    "Contact",
    "Introduction",
    "Payment method",
    "Privacy note",
    "Return method",
    "Shipping and delivery",
    "Technical issue"
]


In [4]:
def text_formatter(text: str) -> str:
    """Perform minor formatting on text."""
    # .replace('\n', ' ') is good. Chaining .strip() is also good.
    cleaned_text = text.replace("\n", " ").strip()
    
    # Add more cleaning here as needed (e.g., removing multiple spaces)
    # import re
    # cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    return cleaned_text

def open_and_read_pdf(document_title: str, base_dir: Path) -> dict:
    """
    Opens a single PDF, extracts text and metadata from each page.
    
    Args:
        document_title: The name of the document (e.g., "Introduction").
        base_dir: The parent directory containing the PDFs.
        
    Returns:
        A dictionary with the document name and a list of page data.
    """
    
    # 1. FIX: Use robust path handling (Pathlib or forward slashes)
    pdf_path = base_dir / f"{document_title}.pdf"
    
    # Check if the file actually exists before trying to open it
    if not pdf_path.exists():
        print(f"Warning: File not found, skipping: {pdf_path}")
        return {'name': document_title, 'doc': [], 'error': 'File not found'}

    doc = fitz.open(pdf_path)
    pages_and_texts = []

    # 2. FIX: Added internal tqdm, but the outer one is more important
    for page_number, page in enumerate(doc):
        text = page.get_text()
        text = text_formatter(text=text)
        
        # 3. FIX: Use text.split() for accurate word count
        word_count = len(text.split())
        
        # 4. FIX: Removed the flawed sentence count. 
        # Add it back ONLY if you use a real tokenizer (e.g., nltk.sent_tokenize)
        
        pages_and_texts.append({
            "page_number": page_number,
            "page_char_count": len(text),
            "page_word_count": word_count, 
            "page_token_count": len(text) / 4, # This heuristic is fine
            "text": text
        })
    
    doc.close() # Good practice to close the document
    return {'name': document_title, 'doc': pages_and_texts}

In [5]:

DATA_DIR = Path("../data/document")

document_data = [] # Renamed from 'document' to be less ambiguous

# 5. FIX: Added tqdm to the OUTER loop to track overall progress
print(f"Processing {len(document_titles)} PDF documents from {DATA_DIR}...")
for title in tqdm(document_titles, desc="Processing PDFs"):
    
    # 6. FIX: Use .append() for clarity and efficiency
    document_data.append(open_and_read_pdf(title, DATA_DIR))

print("Processing complete.")
print(f"Successfully processed {len(document_data)} documents.")


Processing 8 PDF documents from ..\data\document...


Processing PDFs:   0%|          | 0/8 [00:00<?, ?it/s]

Processing complete.
Successfully processed 8 documents.


Check document (pre - chunking)

In [6]:
document_data[0] # Condition of use

{'name': 'Condition of use',
 'doc': [{'page_number': 0,
   'page_char_count': 3188,
   'page_word_count': 466,
   'page_token_count': 797.0,
   'text': 'Conditions of Use  Welcome to My Shop. BK Company (“we,” “our,” “us,” or “My Shop”) provides website  features, products, and related services when you visit myshop.vn, use My Shop mobile  applications, or use any software provided by BK Company in connection with these  services (collectively, the “My Shop Services”). By using the My Shop Services, you  agree—on behalf of yourself, your household members, and others using any My Shop  Service under your account—to these Conditions of Use. Please read them carefully before  using any My Shop Service. Additional terms may apply when using specific services such  as Your Account, Gift Cards, or mobile applications. In case of inconsistency, those Service  Terms will be controlled.  Your privacy is important to us. Please review our Privacy Policy to understand how we  collect, use, and 

In [7]:
n = len(document_data[1]["doc"])

# for i in range(0,n):
#     document_data[1]["doc"][i]['text'] = document_data[1]["doc"][i]['text'].replace("\u200b", ":")

document_data[1] # Contact

{'name': 'Contact',
 'doc': [{'page_number': 0,
   'page_char_count': 1591,
   'page_word_count': 246,
   'page_token_count': 397.75,
   'text': 'Contact & Customer Support  At My Shop, we value our customers and are committed to providing timely assistance for  any questions or concerns you may have. Our Customer Support team is available to help  with order inquiries, account issues, product information, technical problems, and any other  concerns related to your shopping experience.  How to Contact Us\u200b  You can reach our support team through the following channels:  ●\u200b Email Support: For general inquiries, order issues, or feedback, email us at  support@myshop.vn. Please include your Order ID (if applicable) and a clear  description of your concern to help us respond efficiently.\u200b   ●\u200b Phone Support: Call our customer service hotline at +84-123456789 for urgent  assistance during business hours.\u200b   ●\u200b Help Center: Visit our Help Center on the website fo

In [8]:
document_data[2] # Introduction

{'name': 'Introduction',
 'doc': [{'page_number': 0,
   'page_char_count': 725,
   'page_word_count': 106,
   'page_token_count': 181.25,
   'text': 'Introduction  Welcome to our E-Commerce Website, your easy and convenient destination for online  shopping. Our platform is designed with customers in mind, offering a simple and enjoyable  shopping experience.  Browse through a wide range of products, find what you love quickly, and complete your  purchase securely with just a few clicks. With features like personalized recommendations,  detailed product information, and real-time order tracking, shopping has never been easier  or more enjoyable.  We also provide friendly customer support to help you with any questions or concerns,  ensuring a smooth experience from browsing to delivery. Our goal is to make online  shopping simple, fast, and enjoyable for everyone.'}]}

In [9]:
document_data[3] # Payment method

{'name': 'Payment method',
 'doc': [{'page_number': 0,
   'page_char_count': 1835,
   'page_word_count': 294,
   'page_token_count': 458.75,
   'text': 'At My Shop, we provide a variety of payment methods to make shopping simple and  convenient for all customers. You can pay using major credit and debit cards, including  Visa, MasterCard, American Express, Discover, JCB, China UnionPay (credit cards only),  and Diner’s Club (for U.S. billing addresses). In addition, you can use prepaid cards and  gift cards, such as Visa, MasterCard, or American Express prepaid cards, as well as My  Shop Gift Cards. These options give you flexibility to choose the payment method that best  suits your needs.  We take the security of your payments very seriously. All transactions on My Shop are  processed through secure, encrypted payment systems to ensure your information is  protected. My Shop does not store full credit or debit card details, and all payment  processing follows standard safety practice

In [10]:
document_data[4] # Privacy note

{'name': 'Privacy note',
 'doc': [{'page_number': 0,
   'page_char_count': 2644,
   'page_word_count': 374,
   'page_token_count': 661.0,
   'text': 'Privacy Notice for E-Commerce System  Last Updated: November 3, 2025  At My Shop, we take your privacy seriously. This Privacy Notice explains how our  E-Commerce System capstone project (the “System”) collects, uses, and safeguards the  personal information of its users, including Buyers, Sellers, and Admins. Understanding  how your information is handled helps ensure a safe, secure, and personalized shopping  experience.  1. What Information We Collect  We collect personal information to provide and improve the System’s features and services.  Some information is provided directly by you, while other data is collected automatically  during your interactions with the System. When you register as a Buyer or Seller, we collect  basic account information such as your email address and password. You can also provide  and update profile detai

In [11]:
document_data[5] # Return method

{'name': 'Return method',
 'doc': [{'page_number': 0,
   'page_char_count': 2651,
   'page_word_count': 438,
   'page_token_count': 662.75,
   'text': 'Cancellation & Returns Policy  Last Updated: November 3, 2025  At My Shop, we strive to provide a straightforward and transparent process for cancellations  and returns. This policy explains your options if you wish to cancel an order before it ships,  as well as what to do if an item arrives damaged, defective, or incorrect. Our goal is to make  the process simple, fair, and easy to understand.  1. Order Cancellations  You can cancel an order directly through your My Shop account as long as it has not yet  been shipped. To cancel an order, log in to your account and navigate to the "My Orders"  section. Locate the order you wish to cancel, and if the "Cancel Order" button is visible, click  it to submit your cancellation request. The option to cancel is available only while the order  status is "Created," "Paid," or "Processing". Once 

In [12]:
n = len(document_data[1]["doc"])

for i in range(0,n):
    document_data[6]["doc"][i]['text'] = document_data[6]["doc"][i]['text'].replace("\u200b", ":")

document_data[6] # Shipping and delivery

{'name': 'Shipping and delivery',
 'doc': [{'page_number': 0,
   'page_char_count': 2714,
   'page_word_count': 420,
   'page_token_count': 678.5,
   'text': 'Shipping & Delivery Policy  Last Updated: November 3, 2025  This policy explains how order processing, shipping, and delivery are managed on the My  Shop platform. As My Shop operates as a marketplace, products may be shipped directly by  various individual Sellers, which can affect processing times and shipping methods.  Order Processing:  Order processing refers to the time it takes for a Seller to prepare your purchase for  shipment after payment has been confirmed. Most Sellers process orders within 1–3  business days. Once your order has been confirmed, you will see the order status change  from "Paid" to "Processing" in your account\'s "Order History" section. This status indicates  that the Seller is preparing your items for shipment.  Shipping Methods and Costs:  Shipping costs and estimated delivery times are calculated 

In [13]:
document_data[7]["doc"][0]["text"] += document_data[7]["doc"][1]["text"]

document_data[7]["doc"].pop()
document_data[7] # Technical issue

{'name': 'Technical issue',
 'doc': [{'page_number': 0,
   'page_char_count': 3111,
   'page_word_count': 473,
   'page_token_count': 777.75,
   'text': 'Technical & Device Issues Support  At My Shop, we are committed to ensuring that your shopping experience is smooth and  enjoyable across all devices. Occasionally, you may encounter technical or device-related  issues while browsing or using our services. This guide provides solutions to common  problems and helps you troubleshoot issues efficiently.  If you notice that the website or app is not functioning correctly—such as pages failing to  load, buttons not responding, or error messages appearing—first check your internet  connection to ensure your device is connected to a stable Wi-Fi or mobile network. Using the  latest version of your web browser or mobile app can prevent compatibility issues. Clearing  your browser or app cache may resolve temporary glitches, and restarting your device often  fixes minor software problems. If 

Chunking 

In [16]:
from spacy.lang.en import English

nlp = English()

# Add a sentecnizer pipeline, see
nlp.add_pipe("sentencizer")

def split_sentence(pages_and_texts):
    for item in tqdm(pages_and_texts):
        item["sentences"] = list(nlp(item["text"]).sents)
        #Make sure all sentences are strings (the default type is a spaCy datatype)
        item["sentences"] = [str(sentence) for sentence in item["sentences"]]
        # Count
        item["page_sentence_count_spacy"] = len(item["sentences"])

ModuleNotFoundError: No module named 'spacy'

In [ ]:
def split_list(input_list:list, slice_size: int = 10) -> list[list[str]]:
    return [input_list[i:i+ slice_size] for i in range(0, len(input_list),slice_size)]

test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [ ]:
# Loop through pages and texts and split sentences into chunks
def chunking(pages_and_texts,num_sentence_chunk_size):
    for item in tqdm(pages_and_texts):
        item["sentences_chunks"] = split_list(input_list = item["sentences"], slice_size = num_sentence_chunk_size)
        item["num_chunks"] = len(item["sentences_chunks"])

Condition of use

In [ ]:
pages_and_texts = document_data[0]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 2/2 [00:00<?, ?it/s]


[{'page_number': 0,
  'page_char_count': 3188,
  'page_word_count': 466,
  'page_token_count': 797.0,
  'text': 'Conditions of Use  Welcome to My Shop. BK Company (“we,” “our,” “us,” or “My Shop”) provides website  features, products, and related services when you visit myshop.vn, use My Shop mobile  applications, or use any software provided by BK Company in connection with these  services (collectively, the “My Shop Services”). By using the My Shop Services, you  agree—on behalf of yourself, your household members, and others using any My Shop  Service under your account—to these Conditions of Use. Please read them carefully before  using any My Shop Service. Additional terms may apply when using specific services such  as Your Account, Gift Cards, or mobile applications. In case of inconsistency, those Service  Terms will be controlled.  Your privacy is important to us. Please review our Privacy Policy to understand how we  collect, use, and protect your information when you use My 

Contact


In [ ]:
pages_and_texts = document_data[1]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,13)
pages_and_texts

100%|██████████| 1/1 [00:00<00:00, 999.60it/s]


[{'page_number': 0,
  'page_char_count': 1591,
  'page_word_count': 246,
  'page_token_count': 397.75,
  'text': 'Contact & Customer Support  At My Shop, we value our customers and are committed to providing timely assistance for  any questions or concerns you may have. Our Customer Support team is available to help  with order inquiries, account issues, product information, technical problems, and any other  concerns related to your shopping experience.  How to Contact Us  You can reach our support team through the following channels:  ● Email Support: For general inquiries, order issues, or feedback, email us at  support@myshop.vn. Please include your Order ID (if applicable) and a clear  description of your concern to help us respond efficiently.   ● Phone Support: Call our customer service hotline at +84-123456789 for urgent  assistance during business hours.   ● Help Center: Visit our Help Center on the website for FAQs, guides, and tutorials on  account management, orders, return

Introduction


In [ ]:
pages_and_texts = document_data[2]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 1/1 [00:00<00:00, 999.60it/s]


[{'page_number': 0,
  'page_char_count': 725,
  'page_word_count': 106,
  'page_token_count': 181.25,
  'text': 'Introduction  Welcome to our E-Commerce Website, your easy and convenient destination for online  shopping. Our platform is designed with customers in mind, offering a simple and enjoyable  shopping experience.  Browse through a wide range of products, find what you love quickly, and complete your  purchase securely with just a few clicks. With features like personalized recommendations,  detailed product information, and real-time order tracking, shopping has never been easier  or more enjoyable.  We also provide friendly customer support to help you with any questions or concerns,  ensuring a smooth experience from browsing to delivery. Our goal is to make online  shopping simple, fast, and enjoyable for everyone.',
  'sentences': ['Introduction  Welcome to our E-Commerce Website, your easy and convenient destination for online  shopping.',
   'Our platform is designed wit

Payment method

In [ ]:
pages_and_texts = document_data[3]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 1/1 [00:00<?, ?it/s]


[{'page_number': 0,
  'page_char_count': 1835,
  'page_word_count': 294,
  'page_token_count': 458.75,
  'text': 'At My Shop, we provide a variety of payment methods to make shopping simple and  convenient for all customers. You can pay using major credit and debit cards, including  Visa, MasterCard, American Express, Discover, JCB, China UnionPay (credit cards only),  and Diner’s Club (for U.S. billing addresses). In addition, you can use prepaid cards and  gift cards, such as Visa, MasterCard, or American Express prepaid cards, as well as My  Shop Gift Cards. These options give you flexibility to choose the payment method that best  suits your needs.  We take the security of your payments very seriously. All transactions on My Shop are  processed through secure, encrypted payment systems to ensure your information is  protected. My Shop does not store full credit or debit card details, and all payment  processing follows standard safety practices. This helps reduce the risk of unauth

Privacy note

In [ ]:
pages_and_texts = document_data[4]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 3/3 [00:00<?, ?it/s]


[{'page_number': 0,
  'page_char_count': 2644,
  'page_word_count': 374,
  'page_token_count': 661.0,
  'text': 'Privacy Notice for E-Commerce System  Last Updated: November 3, 2025  At My Shop, we take your privacy seriously. This Privacy Notice explains how our  E-Commerce System capstone project (the “System”) collects, uses, and safeguards the  personal information of its users, including Buyers, Sellers, and Admins. Understanding  how your information is handled helps ensure a safe, secure, and personalized shopping  experience.  1. What Information We Collect  We collect personal information to provide and improve the System’s features and services.  Some information is provided directly by you, while other data is collected automatically  during your interactions with the System. When you register as a Buyer or Seller, we collect  basic account information such as your email address and password. You can also provide  and update profile details, including multiple delivery addre

Return method

In [ ]:
pages_and_texts = document_data[5]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 2/2 [00:00<00:00, 2003.49it/s]


[{'page_number': 0,
  'page_char_count': 2651,
  'page_word_count': 438,
  'page_token_count': 662.75,
  'text': 'Cancellation & Returns Policy  Last Updated: November 3, 2025  At My Shop, we strive to provide a straightforward and transparent process for cancellations  and returns. This policy explains your options if you wish to cancel an order before it ships,  as well as what to do if an item arrives damaged, defective, or incorrect. Our goal is to make  the process simple, fair, and easy to understand.  1. Order Cancellations  You can cancel an order directly through your My Shop account as long as it has not yet  been shipped. To cancel an order, log in to your account and navigate to the "My Orders"  section. Locate the order you wish to cancel, and if the "Cancel Order" button is visible, click  it to submit your cancellation request. The option to cancel is available only while the order  status is "Created," "Paid," or "Processing". Once the order status changes to  "Shipped,

Shipping and delivery

In [ ]:
pages_and_texts = document_data[6]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 1/1 [00:00<?, ?it/s]


[{'page_number': 0,
  'page_char_count': 2714,
  'page_word_count': 420,
  'page_token_count': 678.5,
  'text': 'Shipping & Delivery Policy  Last Updated: November 3, 2025  This policy explains how order processing, shipping, and delivery are managed on the My  Shop platform. As My Shop operates as a marketplace, products may be shipped directly by  various individual Sellers, which can affect processing times and shipping methods.  Order Processing:  Order processing refers to the time it takes for a Seller to prepare your purchase for  shipment after payment has been confirmed. Most Sellers process orders within 1–3  business days. Once your order has been confirmed, you will see the order status change  from "Paid" to "Processing" in your account\'s "Order History" section. This status indicates  that the Seller is preparing your items for shipment.  Shipping Methods and Costs:  Shipping costs and estimated delivery times are calculated during checkout and are  determined by the Sel

Technical issue

In [ ]:
pages_and_texts = document_data[7]['doc']
split_sentence(pages_and_texts)
chunking(pages_and_texts,10)
pages_and_texts

100%|██████████| 1/1 [00:00<?, ?it/s]


[{'page_number': 0,
  'page_char_count': 3111,
  'page_word_count': 473,
  'page_token_count': 777.75,
  'text': 'Technical & Device Issues Support  At My Shop, we are committed to ensuring that your shopping experience is smooth and  enjoyable across all devices. Occasionally, you may encounter technical or device-related  issues while browsing or using our services. This guide provides solutions to common  problems and helps you troubleshoot issues efficiently.  If you notice that the website or app is not functioning correctly—such as pages failing to  load, buttons not responding, or error messages appearing—first check your internet  connection to ensure your device is connected to a stable Wi-Fi or mobile network. Using the  latest version of your web browser or mobile app can prevent compatibility issues. Clearing  your browser or app cache may resolve temporary glitches, and restarting your device often  fixes minor software problems. If the issue persists, try accessing My Sho

In [ ]:
document_data

[{'name': 'Condition of use',
  'doc': [{'page_number': 0,
    'page_char_count': 3188,
    'page_word_count': 466,
    'page_token_count': 797.0,
    'text': 'Conditions of Use  Welcome to My Shop. BK Company (“we,” “our,” “us,” or “My Shop”) provides website  features, products, and related services when you visit myshop.vn, use My Shop mobile  applications, or use any software provided by BK Company in connection with these  services (collectively, the “My Shop Services”). By using the My Shop Services, you  agree—on behalf of yourself, your household members, and others using any My Shop  Service under your account—to these Conditions of Use. Please read them carefully before  using any My Shop Service. Additional terms may apply when using specific services such  as Your Account, Gift Cards, or mobile applications. In case of inconsistency, those Service  Terms will be controlled.  Your privacy is important to us. Please review our Privacy Policy to understand how we  collect, use

In [ ]:
document_data

[{'name': 'Condition of use',
  'doc': [{'page_number': 0,
    'page_char_count': 3188,
    'page_word_count': 466,
    'page_token_count': 797.0,
    'text': 'Conditions of Use  Welcome to My Shop. BK Company (“we,” “our,” “us,” or “My Shop”) provides website  features, products, and related services when you visit myshop.vn, use My Shop mobile  applications, or use any software provided by BK Company in connection with these  services (collectively, the “My Shop Services”). By using the My Shop Services, you  agree—on behalf of yourself, your household members, and others using any My Shop  Service under your account—to these Conditions of Use. Please read them carefully before  using any My Shop Service. Additional terms may apply when using specific services such  as Your Account, Gift Cards, or mobile applications. In case of inconsistency, those Service  Terms will be controlled.  Your privacy is important to us. Please review our Privacy Policy to understand how we  collect, use

Embedding

In [ ]:
from langchain_core.documents import Document

# --- Tunable Parameters ---
# You can experiment with these values
CHUNK_SIZE_IN_SENTENCES = 7  # How many sentences to include in one chunk
OVERLAP_IN_SENTENCES = 2     # How many sentences to overlap between chunks

# This will be our new, flat list for the vector store
all_chunks_for_vectorstore = []

# Your document_data list
for doc in document_data:
    doc_name = doc['name']
    
    # Iterate through each page in the document
    for page in doc['doc']:
        page_num = page['page_number']
        
        # --- This is the key change ---
        # We get the flat list of all sentences on the page
        sentences = page['sentences'] 
        
        if not sentences:
            continue

        # Calculate the step size (how many sentences to "slide" forward)
        # If chunk is 7 and overlap is 2, we "slide" 5 sentences
        step = CHUNK_SIZE_IN_SENTENCES - OVERLAP_IN_SENTENCES
        
        # --- Sliding Window Logic ---
        for i in range(0, len(sentences), step):
            
            # Get the chunk of sentences for this window
            chunk_sentences = sentences[i : i + CHUNK_SIZE_IN_SENTENCES]
            
            if not chunk_sentences:
                continue

            # Join the sentences back together to form the chunk's content
            chunk_text = " ".join(chunk_sentences)
            
            # Create the metadata for this specific chunk
            chunk_metadata = {
                "source_document": doc_name,
                "page": page_num,
                "chunk_start_sentence_index": i # Good for debugging
            }
            
            # Create the final Document object
            new_doc = Document(
                page_content=chunk_text,
                metadata=chunk_metadata
            )
            
            all_chunks_for_vectorstore.append(new_doc)

print(f"Created {len(all_chunks_for_vectorstore)} total OVERLAPPING chunks.")



Created 49 total OVERLAPPING chunks.


In [ ]:
import os
from pathlib import Path  # Standard library in Python 3
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

# 1. Define base directory relative to this script file
# This points to the folder containing this script
current_dir = Path(__file__).resolve().parent 

# 2. Define the persist directory dynamically
# This creates: .../backend_ecommerce/ai-module/chroma_db_data
persist_directory = current_dir / "chroma_db_data"

# Ensure the directory exists (good practice)
persist_directory.mkdir(parents=True, exist_ok=True)

print(f"Persisting data to: {persist_directory}")

# --- Configuration ---
if not os.environ.get("GOOGLE_API_KEY"):
    print("Warning: GOOGLE_API_KEY not found in environment variables.")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

collection_name = "rag_document"

# Note: Convert Path object to string for libraries that expect strings
vectorstore = Chroma.from_documents(
    documents=all_chunks_for_vectorstore, 
    embedding=embeddings,
    persist_directory=str(persist_directory), # Convert Path to string here
    collection_name=collection_name
)

print("Embedding and storage complete!")

Embedding and storage complete!
